# Agentic RAG Prototípus

Ez a notebook egy LangGraph-ra épülő Agentic RAG architektúra működését mutatja be.

**Technológiai döntés (LLM):** A kezdetben tesztelt felhős szolgáltatások (Google Gemini Free Tier) API korlátozásai miatt a végleges prototípus a Hugging Face Inference API-t és a **Qwen 2.5 7B** nyílt forráskódú modellt alkalmaztam. A LangChain és a LangGraph absztrakcióinak köszönhetően ez az átállás bizonyítja a rendszer modularitását és autonóm döntéshozatali (routing) képességét.

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    raise ValueError("Hiányzik a HUGGINGFACEHUB_API_TOKEN!")
else:
    print("Hugging Face API token sikeresen betöltve.")

Hugging Face API token sikeresen betöltve.


## 1. Tudásbázis felépítése 

Ebben a blokkban történik a RAG pipeline adatforrásának előkészítése. A script a beolvasott PDF-et kisebb, átfedéssel rendelkező blokkokra darabolja, így elkerülve a fontos mondatok és kontextusok kettévágását.

**Technológiai döntés (Embedding):** Bár a szöveggenerálásért (LLM) egy felhős Hugging Face modell (Qwen 2.5) a felelős, a beágyazási folyamatot egy lokális modellre (`all-MiniLM-L6-v2`) bíztam. Ennek a hibrid megközelítésnek az az oka, hogy a Cloud API-k ingyenes csomagjai szigorú kérés-korlátozással rendelkeznek, ami egy hosszabb dokumentum hálózaton keresztüli vektorizálásánál azonnal hibára futna. A gyors visszakeresést a rendszer egy memóriában futó FAISS vektoradatbázissal biztosítja.

In [8]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

file_path = "data/ZsigoB_irodalomkutatas.pdf"
try:
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    print(f"Betöltve {len(docs)} oldal a dokumentumból.")
except Exception as e:
    print(f"Hiba a fájl beolvasásakor: {e}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200 
)
splits = text_splitter.split_documents(docs)
print(f"A dokumentum {len(splits)} darabra lett felosztva.")

print("Modell betöltése")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
print("Vektoradatbázis betöltve.")

Betöltve 45 oldal a dokumentumból.
A dokumentum 130 darabra (chunk) lett felosztva.
Lokális HuggingFace beágyazó modell betöltése (ez eltarthat pár másodpercig)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vektoradatbázis és retriever sikeresen létrehozva és memóriába töltve!


## 2. Agentic Workflow implementálása

A projekt célja, hogy a chatbot autonóm döntéshozatalt és dinamikus kontextus-kezelést demonstráljon.

**Architekturális felépítés:**
1. **Dinamikus Téma-kinyerés:** A rendszer inicializáláskor önállóan feldolgozza a dokumentum bevezető oldalait, és egy LLM-hívással meghatározza a globális szakmai témát.
2. **Állapot:** Az ágens egy közös szótárban (`AgentState`) vezeti az aktuális kérdést, a kinyert kontextust és a válaszüzenetet.
3. **LLM Router:** A gráf feltételes belépési pontja. Egy szemantikai osztályozó segítségével megvizsgálja, hogy a kérdés kapcsolódik-e a dinamikusan felismert témához.
4. **Retrieve Node:** Releváns kérdés esetén a FAISS adatbázisból kinyeri a kontextust, amit a generáló egység (Qwen 2.5 7B) a hallucinációk szigorú elkerülésével dolgoz fel.
5. **Biztonsági Elutasítás:** Irreleváns kérdés esetén az ágens egyáltalán nem égeti az LLM erőforrásait: azonnali, hardkódolt elutasítással védi a rendszert a témán kívüli beszélgetésektől.

In [19]:
import os
from typing import TypedDict, List
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    question: str
    context: List[str]
    answer: str

endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.1, 
    huggingfacehub_api_token=os.environ.get("HUGGINGFACEHUB_API_TOKEN")
)
llm = ChatHuggingFace(llm=endpoint)

print("Dokumentum témájának elemzése")
intro_text = "\n".join([docs[0].page_content, docs[4].page_content, docs[5].page_content])[:1500]
topic_prompt = f"Kérlek határozd meg maximum 5 szóban, hogy mi a pontos szakmai témája a következő dokumentumnak: {intro_text}"
DOCUMENT_TOPIC = llm.invoke(topic_prompt).content.strip()
print(f"Téma: {DOCUMENT_TOPIC}")

def retrieve_node(state: AgentState):
    print("Releváns részek keresése")
    documents = retriever.invoke(state["question"])
    context = [doc.page_content for doc in documents]
    return {"context": context}

def generate_node(state: AgentState):
    print("Válasz generálása")
    question = state["question"]
    
    if state.get("context"):
        context_str = "\n\n".join(state["context"])
        prompt = f"Kérlek, kizárólag a következő kontextus alapján válaszolj a kérdésre magyar nyelven! Ha nincs benne a válasz, írd azt: 'Nincs erről információm.'\n\nKontextus: {context_str}\n\nKérdés: {question}"
        response = llm.invoke(prompt)
        return {"answer": response.content}
    else:
        return {"answer": f"Sajnálom, de ez a kérdés nem tűnik relevánsnak a betöltött dokumentum ({DOCUMENT_TOPIC}) témájához. Kérlek, kérdezz a dokumentummal kapcsolatban!"}

def route_question(state: AgentState):
    print("\nKérdés elemzése")
    question = state["question"]
    
    router_prompt = f"""Eldöntendő feladat: A felhasználó kérdése kapcsolódik-e a(z) '{DOCUMENT_TOPIC}' témakörhöz, VAGY annak bármilyen tudományos hátteréhez, elméleti alapozásához és kapcsolódó módszertanához?
    Kérdés: {question}
    Ha igen, vagy ha bizonytalan vagy, válaszolj 'szakmai'-t. Ha egyértelműen teljesen más, válaszolj 'egyéb'-et.
    Csak egyetlen szót válaszolj: 'szakmai' vagy 'egyéb'."""
    
    decision = llm.invoke(router_prompt).content.lower()
    
    if "szakmai" in decision:
        print(f"Besorolás: szakmai ({decision.strip()})")
        return "retrieve"
    else:
        print(f"Besorolás: egyéb ({decision.strip()})")
        return "generate"

workflow = StateGraph(AgentState)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)

workflow.set_conditional_entry_point(
    route_question,
    {
        "retrieve": "retrieve",
        "generate": "generate"
    }
)
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()
print("Agentic RAG workflow kész")

Dokumentum témájának elemzése
Téma: Gépi tanulás és erősített tanulás
Agentic RAG workflow kész


In [20]:
print("Jöhetnek a kérdések (kilépés: exit vagy kilepes)")
print("-" * 30)

while True:
    user_input = input("\nKérdés: ")
    
    if user_input.lower() in ['exit', 'kilepes', 'quit']:
        print("Szia!")
        break
        
    if not user_input.strip():
        continue
        
    inputs = {"question": user_input, "context": []}
    
    for output in app.stream(inputs):
        pass 
        
    print(f"\nChatbot: {output['generate']['answer']}")
    print("-" * 50)

Jöhetnek a kérdések (kilépés: exit vagy kilepes)
------------------------------



Kérdés:  mik a Diszkrét módszerek korlátai folytonos környezetben



Kérdés elemzése
Besorolás: szakmai (szakmai)
Releváns részek keresése
Válasz generálása

Chatbot: A diszkrét módszerek korlátai folytonos környezetben a következők:

1. **Végtelen állapottér**: A folytonos állapotok száma végtelen, ami lehetetlenné teszi a Q-táblázat explicit felépítését és tárolását.
2. **Generalizáció hiánya**: Ha diszkretizálnánk a folytonos teret, az állapottér exponenciálisan növekedne a dimenziók számával. Ráadásul az ágensnek minden egyes diszkretizált „cellában” külön kellene tanulnia, és nem tudna generalizálni a szomszédos cellákra.
--------------------------------------------------



Kérdés:  exit


Szia!


## 3. Bottlenecks és Tesztelés

A prototípus fejlesztése és tesztelése során az alábbi architekturális kihívásokat azonosítottam és kezeltem:

### 1. Bottlenecks és Megoldásuk
* **API Instabilitás és Rate Limiting:** A felhős (Free tier) API-k szigorú kérés-korlátai instabillá tették a tesztelést. A LangChain wrapper architektúra (ChatHuggingFace) lehetővé tette a modellek azonnali cseréjét, így a rendszer hibatűrő maradt.
* **Túlbuzgóság és Hallucináció:** A nyelvi modellek hajlamosak a témán kívüli kérdésekre is kitalált válaszokat adni. Ezt egy hibrid router-architektúrával küszöböltem ki: az irreleváns kérdéseknél az ágens megkerüli az LLM-et, és determinisztikus Python logikával utasítja el a válaszadást.

### 2. A Teljesítmény Mérési és Tesztelési Stratégiája
Egy éles rendszerben a manuális ellenőrzést az alábbi automatizált metrikák váltanák fel:
* **Válaszok Minősége (RAGAS):** A Faithfulness (a modell csak a kontextusból válaszol-e) és az Answer Relevance (mennyire releváns a válasz) metrikák mérésével biztosítható a kimenet megbízhatósága.

### 3. Továbbfejlesztési Javaslatok
1. **Query Rewriting:** Egy újabb LangGraph csomópont beépítése, amely a felhasználó pontatlan kérdését (például rövidítés) a FAISS adatbázis számára optimális formátumra írja át a keresés előtt.
2. **Lokális LLM:** A felhős API-k kiváltása egy dedikált szerveren futó `vLLM` példánnyal.